# Network metrics: connected components

**Algorithm:** `compute_connected_components` from `zarr_vectors_tools.algorithms`

Chunked union-find over a `graph` store: per-chunk DSU on intra-chunk edges, then a single global pass over the cross-chunk array. Scales beyond memory because per-chunk edge arrays are loaded one at a time.

1. Generate a synthetic network with several disconnected sub-graphs
2. Write the graph
3. Run `compute_connected_components`
4. Inspect the component-size distribution
5. Persist labels by re-running `write_graph` with labels in `node_attributes` (write-back to existing stores requires core *Add 5*)

In [1]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import open_zvr
from zarr_vectors.types.graphs import write_graph, read_graph
from zarr_vectors_tools.algorithms import compute_connected_components

_tmpdir = tempfile.mkdtemp(prefix="zvt_examples_")
STORE         = os.path.join(_tmpdir, "components.zarrvectors")
STORE_LABELED = os.path.join(_tmpdir, "components_labeled.zarrvectors")
print("Stores:", STORE, STORE_LABELED)

Stores: C:\Users\andre\AppData\Local\Temp\zvt_examples_s3ei0lko\components.zarrvectors C:\Users\andre\AppData\Local\Temp\zvt_examples_s3ei0lko\components_labeled.zarrvectors


## 1 · Generate a network with several disconnected sub-graphs

Spawn four spatially separated clusters and connect each cluster internally as a k-NN graph. The clusters share no edges, so the ground-truth component count is 4.

In [2]:
from scipy.spatial import KDTree

rng = np.random.default_rng(7)
cluster_centres = np.array([
    [100., 100., 100.],
    [400., 100., 100.],
    [100., 400., 400.],
    [400., 400., 400.],
], dtype=np.float32)
n_per_cluster = 600
n_nodes = len(cluster_centres) * n_per_cluster

positions = np.concatenate([
    centre + rng.normal(0, 25, (n_per_cluster, 3)).astype(np.float32)
    for centre in cluster_centres
]).astype(np.float32)

# k-NN within each cluster (no cross-cluster edges).
edges_all = []
for ci in range(len(cluster_centres)):
    start = ci * n_per_cluster
    block = positions[start:start + n_per_cluster]
    tree = KDTree(block)
    _, nbrs = tree.query(block, k=5)         # k=1 is self
    src = np.repeat(np.arange(n_per_cluster, dtype=np.int32), 4) + start
    dst = nbrs[:, 1:].flatten().astype(np.int32) + start
    edges_all.append(np.column_stack([src, dst]))
edges = np.concatenate(edges_all, axis=0)

# Canonicalise and dedupe.
mask  = edges[:, 0] != edges[:, 1]
edges = edges[mask]
edges = np.sort(edges, axis=1)
edges = np.unique(edges, axis=0).astype(np.int32)

print(f"nodes      : {n_nodes:,}")
print(f"edges      : {len(edges):,}")
print(f"clusters   : {len(cluster_centres)} (ground truth)")
print(f"bounds     : lo={positions.min(0).round(1)} hi={positions.max(0).round(1)}")

nodes      : 2,400
edges      : 6,206
clusters   : 4 (ground truth)
bounds     : lo=[11.8  8.5 13.7] hi=[477.1 501.5 480.3]


## 2 · Write the graph

Pick a `chunk_shape` smaller than the cluster spread so the algorithm has to fuse pieces across chunks (this exercises the cross-chunk union step).

In [3]:
write_graph(
    STORE,
    positions=positions,
    edges=edges,
    chunk_shape=(100., 100., 100.),
    bin_shape=(25., 25., 25.),
    is_tree=False,
)
store = open_zvr(STORE)
print(f"vertex_count : {store[0].vertex_count:,}")
print(f"chunk_shape  : {store.chunk_shape}")

vertex_count : 2,400
chunk_shape  : (100.0, 100.0, 100.0)


## 3 · Run `compute_connected_components`

Returns `labels` in the global vertex ordering (matches `read_graph`'s positions), plus a per-component size table.

In [4]:
cc = compute_connected_components(STORE)
labels = cc["labels"]
print(f"n_components            : {cc['n_components']}")
print(f"largest_component_size  : {cc['largest_component_size']:,}")
print(f"labels.shape, dtype     : {labels.shape}, {labels.dtype}")

n_components            : 4
largest_component_size  : 600
labels.shape, dtype     : (2400,), uint32


## 4 · Inspect the component-size distribution

Sort the sizes table descending and print the top components. With four k-NN clusters of equal size we expect four components of roughly `n_per_cluster` nodes each (small variance because the k-NN graph is occasionally not fully internally connected).

In [5]:
sizes_sorted = np.sort(cc["component_sizes"])[::-1]
print("Top components by size:")
for rank, sz in enumerate(sizes_sorted[:10]):
    bar = "#" * int(40 * sz / sizes_sorted[0])
    print(f"  #{rank:>2}  {sz:>5}  {bar}")

tail = int(sizes_sorted[10:].sum()) if len(sizes_sorted) > 10 else 0
print(f"\nRemaining (rank > 10) nodes: {tail:,}")

# Cross-check: the labels must partition the vertex set without gaps.
assert labels.min() == 0 and labels.max() == cc['n_components'] - 1
assert int(cc['component_sizes'].sum()) == len(labels)

Top components by size:
  # 0    600  ########################################
  # 1    600  ########################################
  # 2    600  ########################################
  # 3    600  ########################################

Remaining (rank > 10) nodes: 0


## 5 · Persist the labels in a new store

`write_back=True` raises `NotImplementedError` — post-hoc per-node attribute writes need core *Add 5*. Until that lands, the idiomatic way to keep the labels is to re-write the graph with `labels` as a node attribute.

In [6]:
# Read positions+edges back in the same vertex ordering the labels use.
g = read_graph(STORE)

write_graph(
    STORE_LABELED,
    positions=g["positions"],
    edges=g["edges"],
    chunk_shape=(100., 100., 100.),
    bin_shape=(25., 25., 25.),
    is_tree=False,
    node_attributes={"component": labels.astype(np.int32)},
)

# read_graph returns positions + edges only; for node attributes go
# through the lazy API, which materialises them in the same global
# vertex ordering.
store_labeled = open_zvr(STORE_LABELED)
comp = store_labeled[0].attributes["component"].compute()
print(f"Persisted component labels: shape={comp.shape}, dtype={comp.dtype}")
print(f"Unique component ids in store: {sorted(set(comp.tolist()))[:10]}")

Persisted component labels: shape=(2400,), dtype=int32
Unique component ids in store: [0, 1, 2, 3]


## Summary

| Step | API |
|------|-----|
| Compute components | `compute_connected_components(store, level=0)` |
| Returns | `{labels, n_components, largest_component_size, component_sizes}` |
| Persist labels | Re-run `write_graph(..., node_attributes={'component': labels})` |

> **Limitation tracked as core *Add 5*:** `write_back=True` cannot patch labels into an existing store yet.